# Nested Sampling in the General Gaussian Case Using MCMC

This notebook explains how to move from the **special case**, where we could sample exactly from the constrained prior, to the **general case**, where we instead use an **MCMC kernel** to sample approximately from

$$
\pi(	\theta \mid L(	\theta) > \ell).
$$

The goal is to answer the following points clearly:

1. why an MCMC sampler can target the constrained prior,
2. how this MCMC step is inserted into nested sampling,
3. how to implement the resulting algorithm,
4. how to run a few small numerical checks.

We keep the presentation practical and implementation-oriented.

## 1. Imports

We use:

- `numpy` for linear algebra and simulation,
- `scipy.special.logsumexp` for stable log-sums,
- `matplotlib` for diagnostics.

The notebook works in the Gaussian Bayesian model

$$
y_i \mid 	\theta \sim \mathcal N_d(	\theta,\Sigma), \qquad
	\theta \sim \mathcal N_d(0,I_d),
$$

with known covariance matrix `Sigma`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import logsumexp

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 2. General Gaussian model: data generation and exact benchmark

We work in the general Gaussian model, not in the simplified radial case.

For the performance study, it is still useful to keep a benchmark.  
Since this model is conjugate, the marginal likelihood has a closed form, so we can compare the nested sampling estimate with the exact `log-evidence`.

In [ ]:
def simulate_gaussian_data(n, d, Sigma, theta_true=None, seed=None):
    """
    Simulate y_1, ..., y_n in R^d from N_d(theta_true, Sigma).
    """
    rng = np.random.default_rng(seed)
    if theta_true is None:
        theta_true = np.zeros(d)
    Y = rng.multivariate_normal(mean=theta_true, cov=Sigma, size=n)
    return Y


def posterior_parameters(Y, Sigma):
    """
    Posterior parameters for:
        y_i | theta ~ N_d(theta, Sigma)
        theta ~ N_d(0, I_d)
    """
    n, d = Y.shape
    Sigma_inv = np.linalg.inv(Sigma)
    Vn = np.linalg.inv(np.eye(d) + n * Sigma_inv)
    mn = Vn @ Sigma_inv @ np.sum(Y, axis=0)
    return mn, Vn


def true_log_evidence_general(Y, Sigma):
    """
    Closed-form log marginal likelihood in the general Gaussian model.
    """
    n, d = Y.shape
    Sigma_inv = np.linalg.inv(Sigma)

    A = np.eye(d) + n * Sigma_inv
    b = Sigma_inv @ np.sum(Y, axis=0)

    sign_Sigma, logdet_Sigma = np.linalg.slogdet(Sigma)
    sign_A, logdet_A = np.linalg.slogdet(A)
    if sign_Sigma <= 0 or sign_A <= 0:
        raise ValueError("Sigma and A must be positive definite.")

    quad1 = sum(y @ Sigma_inv @ y for y in Y)
    quad2 = b @ np.linalg.solve(A, b)

    logZ = (
        -0.5 * n * d * np.log(2 * np.pi)
        -0.5 * n * logdet_Sigma
        -0.5 * logdet_A
        -0.5 * (quad1 - quad2)
    )
    return logZ

## 3. Prior density and likelihood

In the general case, the likelihood is no longer radial in $\theta$, so we cannot reduce the constraint to a simple ball.

But we can still evaluate:

- the log-prior,
- the log-likelihood.

These are the only ingredients needed for a Metropolis-Hastings kernel.

In [ ]:
def log_prior(theta):
    """
    Log-density of N_d(0, I_d), up to the normalizing constant.
    For MH with a symmetric proposal, the additive constant cancels.
    """
    return -0.5 * np.dot(theta, theta)


def loglik_general(theta, Y, Sigma):
    """
    Log-likelihood of Y given theta in the Gaussian model:
        y_i | theta ~ N_d(theta, Sigma).
    """
    n, d = Y.shape
    Sigma_inv = np.linalg.inv(Sigma)
    sign, logdet = np.linalg.slogdet(Sigma)
    if sign <= 0:
        raise ValueError("Sigma must be positive definite.")

    centered = Y - theta
    quad = sum(v @ Sigma_inv @ v for v in centered)

    return -0.5 * n * d * np.log(2 * np.pi) - 0.5 * n * logdet - 0.5 * quad

## 4. Why an MCMC kernel can target the constrained prior

The target distribution we need inside nested sampling is

$$
\pi_\ell(\theta) \propto \pi(\theta)\,\mathbf{1}\{L(\theta) > \ell\}.
$$

This is simply the prior restricted to the admissible region.

A convenient idea is to use a **random-walk Metropolis-Hastings** proposal:

$$
\theta' = \theta + \varepsilon,
\qquad
\varepsilon \sim \mathcal{N}_d(0,\sigma_{\mathrm{prop}}^2 I_d).
$$

Because the proposal is symmetric, the Metropolis-Hastings ratio only involves the prior, provided the proposal satisfies the likelihood constraint. So the move is:

- if $L(\theta') \le \ell$, reject immediately,
- otherwise accept with probability

$$
\alpha(\theta,\theta')
=
\min\left(1,\frac{\pi(\theta')}{\pi(\theta)}\right).
$$

This leaves the truncated prior invariant.

So even though we cannot sample exactly from the constrained prior, we can sample approximately from it by running this Markov chain long enough.

## 5. Constrained Metropolis-Hastings kernel

The next function performs several Metropolis-Hastings steps targeting

$$
\pi(	\theta \mid L(	\theta) > \ell).
$$

A good initial state is any point that already satisfies the constraint.  
Inside nested sampling, a surviving live point is perfect for that.

In [ ]:
def constrained_mh_sampler(
    theta0,
    logL_min,
    Y,
    Sigma,
    n_steps=50,
    proposal_scale=0.5,
    seed=None,
    return_trace=False
):
    """
    Run a random-walk MH chain targeting the constrained prior:
        pi(theta | L(theta) > L_min).

    Proposal:
        theta' = theta + proposal_scale * z,  z ~ N(0, I)

    Acceptance rule:
    - reject automatically if loglik(theta') <= logL_min
    - otherwise accept with MH ratio based on the prior only
      because the proposal is symmetric and the indicator constraint
      is enforced by rejection.
    """
    rng = np.random.default_rng(seed)
    theta = np.array(theta0, dtype=float).copy()
    d = theta.shape[0]

    current_logprior = log_prior(theta)
    current_loglik = loglik_general(theta, Y, Sigma)

    if current_loglik <= logL_min:
        raise ValueError("Initial state does not satisfy the likelihood constraint.")

    accepted = 0
    trace = [theta.copy()] if return_trace else None

    for _ in range(n_steps):
        proposal = theta + proposal_scale * rng.normal(size=d)
        proposal_loglik = loglik_general(proposal, Y, Sigma)

        # Constraint enforcement
        if proposal_loglik <= logL_min:
            if return_trace:
                trace.append(theta.copy())
            continue

        proposal_logprior = log_prior(proposal)
        log_alpha = proposal_logprior - current_logprior

        if np.log(rng.uniform()) < min(0.0, log_alpha):
            theta = proposal
            current_logprior = proposal_logprior
            current_loglik = proposal_loglik
            accepted += 1

        if return_trace:
            trace.append(theta.copy())

    out = {
        "theta": theta,
        "acceptance_rate": accepted / n_steps
    }

    if return_trace:
        out["trace"] = np.array(trace)

    return out

## 6. Small sanity check for the constrained MCMC sampler

Before inserting the chain into nested sampling, it is good practice to test it separately.

We will:

1. simulate a small dataset,
2. choose a likelihood threshold,
3. start from a point satisfying the constraint,
4. run the constrained MCMC chain,
5. check the acceptance rate and a trace plot.

This does not prove correctness mathematically, but it helps detect obvious implementation mistakes.

In [ ]:
# Small test problem
d_test = 2
n_test = 20
Sigma_test = np.array([[1.0, 0.4],
                       [0.4, 1.5]])

Y_test = simulate_gaussian_data(n_test, d_test, Sigma_test, seed=123)

# Start from a prior draw, then keep trying until it satisfies the constraint
rng = np.random.default_rng(2024)
theta_candidates = rng.normal(size=(1000, d_test))
logliks = np.array([loglik_general(t, Y_test, Sigma_test) for t in theta_candidates])

# Pick a threshold below a reasonably high likelihood level
logL_min_test = np.quantile(logliks, 0.60)

valid_idx = np.where(logliks > logL_min_test)[0][0]
theta0_test = theta_candidates[valid_idx]

mh_out = constrained_mh_sampler(
    theta0_test,
    logL_min_test,
    Y_test,
    Sigma_test,
    n_steps=200,
    proposal_scale=0.4,
    seed=1,
    return_trace=True
)

print("Acceptance rate:", mh_out["acceptance_rate"])
print("Final state satisfies constraint:",
      loglik_general(mh_out["theta"], Y_test, Sigma_test) > logL_min_test)

### Trace of the constrained MCMC chain in dimension 2

Because the test is two-dimensional, we can visualize the trace.

The goal is not to produce a very advanced MCMC diagnostic here, only to see that the chain moves inside the admissible region instead of getting stuck immediately.

In [ ]:
trace = mh_out["trace"]

plt.figure()
plt.plot(trace[:, 0], trace[:, 1], marker="o", markersize=2, linewidth=1)
plt.scatter(trace[0, 0], trace[0, 1], marker="s", s=60, label="Start")
plt.scatter(trace[-1, 0], trace[-1, 1], marker="x", s=80, label="End")
plt.xlabel(r"$\theta_1$")
plt.ylabel(r"$\theta_2$")
plt.title("Trace of constrained MH chain")
plt.legend()
plt.show()

## 7. How to use this MCMC kernel inside nested sampling

The structure of nested sampling remains the same.

At each iteration:

1. keep a set of live points,
2. remove the one with smallest likelihood,
3. update the prior mass estimate,
4. add the dead-point contribution to the evidence,
5. replace the dead point with a new point drawn from the constrained prior.

The only difference from the special case is step 5.

Since we cannot sample exactly from

$$
\pi(	\theta \mid L(	\theta) > L_t),
$$

we do the following:

- choose one surviving live point as an initial state,
- run the constrained MCMC chain targeting that truncated prior,
- use the final state of the chain as the replacement live point.

This is the standard MCMC-within-NS idea.

## 8. Nested sampling with MCMC replacement

The function below implements nested sampling in the general case.

A few practical comments:

- the initial state of the MCMC chain is chosen among the surviving live points, because it automatically satisfies the constraint,
- the parameter `mcmc_steps` controls how long we run the chain before taking the new live point,
- the parameter `proposal_scale` controls the local size of MH moves,
- the final evidence estimate includes the remaining live-point contribution.

In [ ]:
def nested_sampling_general_mcmc(
    Y,
    Sigma,
    N_live=100,
    max_iter=500,
    seed=None,
    use_random_shrinkage=True,
    mcmc_steps=50,
    proposal_scale=0.5,
    stop_tol=1e-10,
    return_path=False
):
    rng = np.random.default_rng(seed)
    n, d = Y.shape

    # Initialize live points from the prior
    live_thetas = rng.normal(size=(N_live, d))
    live_logLs = np.array([loglik_general(theta, Y, Sigma) for theta in live_thetas])

    X_prev = 1.0
    log_terms = []
    dead_logLs = []
    Xs = [X_prev]
    logZ_partial = []
    mh_acceptance_rates = []

    for _ in range(max_iter):
        # 1. Remove worst live point
        worst_idx = np.argmin(live_logLs)
        logL_t = live_logLs[worst_idx]
        dead_logLs.append(logL_t)

        # 2. Shrink prior mass
        if use_random_shrinkage:
            T_t = rng.beta(N_live, 1)
        else:
            T_t = np.exp(-1.0 / N_live)

        X_t = X_prev * T_t
        w_t = X_prev - X_t

        # 3. Add dead-point contribution
        log_terms.append(np.log(w_t) + logL_t)
        logZ_now = logsumexp(log_terms)
        logZ_partial.append(logZ_now)

        # 4. Stop if remaining possible evidence is negligible
        log_remaining_upper = np.log(X_t) + np.max(live_logLs)
        if log_remaining_upper - logZ_now < np.log(stop_tol):
            X_prev = X_t
            Xs.append(X_prev)
            break

        # 5. Choose a surviving live point to start the MCMC chain
        survivor_indices = [i for i in range(N_live) if i != worst_idx]
        start_idx = rng.choice(survivor_indices)
        theta_start = live_thetas[start_idx].copy()

        mh_seed = int(rng.integers(0, 10**9))
        mh_out = constrained_mh_sampler(
            theta0=theta_start,
            logL_min=logL_t,
            Y=Y,
            Sigma=Sigma,
            n_steps=mcmc_steps,
            proposal_scale=proposal_scale,
            seed=mh_seed,
            return_trace=False
        )

        theta_new = mh_out["theta"]
        logL_new = loglik_general(theta_new, Y, Sigma)
        mh_acceptance_rates.append(mh_out["acceptance_rate"])

        live_thetas[worst_idx] = theta_new
        live_logLs[worst_idx] = logL_new

        X_prev = X_t
        Xs.append(X_prev)

    # Remaining live-point contribution
    log_live_meanL = logsumexp(live_logLs) - np.log(N_live)
    logZ_live = np.log(X_prev) + log_live_meanL

    logZ_hat_no_live = logsumexp(log_terms)
    logZ_hat = logsumexp([logZ_hat_no_live, logZ_live])

    out = {
        "logZ_hat": logZ_hat,
        "logZ_hat_no_live": logZ_hat_no_live,
        "dead_logLs": np.array(dead_logLs),
        "Xs": np.array(Xs),
        "live_logLs": live_logLs.copy(),
        "mean_mh_acceptance": np.mean(mh_acceptance_rates) if mh_acceptance_rates else np.nan,
        "mh_acceptance_rates": np.array(mh_acceptance_rates),
    }

    if return_path:
        out["logZ_partial"] = np.array(logZ_partial)

    return out

## 9. One run of nested sampling with MCMC

We now run the full algorithm once on a moderate problem.

The first goal is interpretability:

- compare the final estimate to the exact log-evidence,
- look at the running estimate through time,
- inspect the average acceptance rate of the MCMC replacement step.

In [ ]:
# Example problem
n = 25
d = 3
Sigma = np.array([[1.0, 0.2, 0.1],
                  [0.2, 1.5, 0.3],
                  [0.1, 0.3, 1.2]])

Y = simulate_gaussian_data(n, d, Sigma, seed=42)
logZ_true = true_log_evidence_general(Y, Sigma)

out_one = nested_sampling_general_mcmc(
    Y,
    Sigma,
    N_live=100,
    max_iter=600,
    seed=1,
    use_random_shrinkage=True,
    mcmc_steps=80,
    proposal_scale=0.45,
    stop_tol=1e-10,
    return_path=True
)

print(f"True log-evidence      : {logZ_true:.6f}")
print(f"NS-MCMC estimate       : {out_one['logZ_hat']:.6f}")
print(f"Error                  : {out_one['logZ_hat'] - logZ_true:.6f}")
print(f"Mean MH acceptance rate: {out_one['mean_mh_acceptance']:.3f}")

### Convergence plot for one run

This plot shows how the nested sampling estimate is progressively built.

As before, the dashed line is the exact log-evidence.  
The purpose here is mostly pedagogical: we want to see whether the cumulative estimate stabilizes in a reasonable region.

In [ ]:
plt.figure()
plt.plot(out_one["logZ_partial"], label="Running estimate")
plt.axhline(logZ_true, linestyle="--", linewidth=1.5, label="True log-evidence")
plt.xlabel("Iteration")
plt.ylabel(r"log-evidence estimate")
plt.title("NS with MCMC replacement: one-run convergence")
plt.legend()
plt.show()

### Prior-mass shrinkage

This plot is useful to keep the nested sampling mechanism visible even though the replacement step is now performed by MCMC.

In [ ]:
plt.figure()
plt.plot(-np.log(out_one["Xs"][1:]))
plt.xlabel("Iteration")
plt.ylabel(r"$-\log X_t$")
plt.title("Prior mass shrinkage")
plt.show()

## 10. Repeated runs on one fixed dataset

We now repeat the full NS-MCMC algorithm several times on the same dataset.

The idea is simple: before discussing larger-scale behavior, we first want to know whether the algorithm gives reasonably stable estimates on one concrete problem.

In [ ]:
def assess_ns_mcmc_single_dataset(
    Y,
    Sigma,
    N_live_list=(50, 100, 200),
    n_rep=40,
    max_iter=500,
    mcmc_steps=50,
    proposal_scale=0.5,
    seed=12345
):
    true_logZ = true_log_evidence_general(Y, Sigma)
    rng = np.random.default_rng(seed)
    results = {}

    for N_live in N_live_list:
        estimates = []
        acceptances = []

        for _ in range(n_rep):
            run_seed = int(rng.integers(0, 10**9))
            out = nested_sampling_general_mcmc(
                Y,
                Sigma,
                N_live=N_live,
                max_iter=max_iter,
                seed=run_seed,
                use_random_shrinkage=True,
                mcmc_steps=mcmc_steps,
                proposal_scale=proposal_scale,
                stop_tol=1e-10,
                return_path=False
            )
            estimates.append(out["logZ_hat"])
            acceptances.append(out["mean_mh_acceptance"])

        estimates = np.array(estimates)
        errors = estimates - true_logZ

        results[N_live] = {
            "true_logZ": true_logZ,
            "mean_logZ_hat": np.mean(estimates),
            "std_logZ_hat": np.std(estimates, ddof=1),
            "bias_logZ": np.mean(errors),
            "rmse_logZ": np.sqrt(np.mean(errors**2)),
            "mean_acceptance": np.mean(acceptances),
            "estimates": estimates,
            "errors": errors
        }

    return results

In [ ]:
results_one_dataset = assess_ns_mcmc_single_dataset(
    Y,
    Sigma,
    N_live_list=(50, 100, 200),
    n_rep=40,
    max_iter=500,
    mcmc_steps=80,
    proposal_scale=0.45,
    seed=2024
)

for N_live, res in results_one_dataset.items():
    print(
        f"N_live={N_live:3d} | "
        f"mean={res['mean_logZ_hat']:.6f} | "
        f"std={res['std_logZ_hat']:.6f} | "
        f"bias={res['bias_logZ']:.6f} | "
        f"rmse={res['rmse_logZ']:.6f} | "
        f"mean_acceptance={res['mean_acceptance']:.3f}"
    )

### Boxplots of the repeated estimates

This figure gives a quick visual summary of the repeated-run variability.  
We keep this part short on purpose: the point here is mainly to confirm that the NS-MCMC procedure behaves sensibly.

In [ ]:
N_vals = sorted(results_one_dataset.keys())
data = [results_one_dataset[N]["estimates"] for N in N_vals]
true_val = results_one_dataset[N_vals[0]]["true_logZ"]

plt.figure()
plt.boxplot(data, labels=[str(N) for N in N_vals])
plt.axhline(true_val, linestyle="--", linewidth=1.5, label="True log-evidence")
plt.xlabel("Number of live points")
plt.ylabel("Estimated log-evidence")
plt.title("NS-MCMC estimates across repetitions")
plt.legend()
plt.show()

## 11. Small test on the MCMC budget

In the general case, a key practical question is how many MCMC steps should be used when generating the replacement live point.

We do a small comparison across a few values of `mcmc_steps`, keeping everything else fixed.

This is not meant to be an exhaustive study, only a practical diagnostic.

In [ ]:
def assess_mcmc_budget(
    Y,
    Sigma,
    mcmc_steps_list=(10, 30, 80, 150),
    n_rep=30,
    N_live=100,
    max_iter=500,
    proposal_scale=0.45,
    seed=321
):
    true_logZ = true_log_evidence_general(Y, Sigma)
    rng = np.random.default_rng(seed)
    results = {}

    for mcmc_steps in mcmc_steps_list:
        estimates = []
        acceptances = []

        for _ in range(n_rep):
            run_seed = int(rng.integers(0, 10**9))
            out = nested_sampling_general_mcmc(
                Y,
                Sigma,
                N_live=N_live,
                max_iter=max_iter,
                seed=run_seed,
                mcmc_steps=mcmc_steps,
                proposal_scale=proposal_scale,
                return_path=False
            )
            estimates.append(out["logZ_hat"])
            acceptances.append(out["mean_mh_acceptance"])

        estimates = np.array(estimates)
        errors = estimates - true_logZ

        results[mcmc_steps] = {
            "mean": np.mean(estimates),
            "std": np.std(estimates, ddof=1),
            "bias": np.mean(errors),
            "rmse": np.sqrt(np.mean(errors**2)),
            "mean_acceptance": np.mean(acceptances),
            "estimates": estimates,
        }

    return results

In [ ]:
budget_results = assess_mcmc_budget(
    Y,
    Sigma,
    mcmc_steps_list=(10, 30, 80, 150),
    n_rep=30,
    N_live=100,
    max_iter=500,
    proposal_scale=0.45,
    seed=321
)

for msteps, res in budget_results.items():
    print(
        f"mcmc_steps={msteps:3d} | "
        f"mean={res['mean']:.6f} | "
        f"std={res['std']:.6f} | "
        f"bias={res['bias']:.6f} | "
        f"rmse={res['rmse']:.6f} | "
        f"mean_acceptance={res['mean_acceptance']:.3f}"
    )

### RMSE as a function of the MCMC budget

If increasing the MCMC budget improves the estimate, we should see the RMSE decrease or at least stabilize.  
In practice, the behavior may not be perfectly monotone because the experiment is still Monte Carlo and kept deliberately small.

In [ ]:
m_vals = sorted(budget_results.keys())
rmse_vals = [budget_results[m]["rmse"] for m in m_vals]
acc_vals = [budget_results[m]["mean_acceptance"] for m in m_vals]

plt.figure()
plt.plot(m_vals, rmse_vals, marker="o")
plt.xlabel("MCMC steps per replacement")
plt.ylabel("RMSE of log-evidence")
plt.title("Effect of the MCMC budget")
plt.show()

plt.figure()
plt.plot(m_vals, acc_vals, marker="o")
plt.xlabel("MCMC steps per replacement")
plt.ylabel("Mean MH acceptance rate")
plt.title("Acceptance rate vs MCMC budget")
plt.show()

## 12. Optional small sensitivity test for the proposal scale

Another practical tuning parameter is the proposal scale of the random-walk Metropolis kernel.

If the proposal scale is too small, the chain moves slowly.  
If it is too large, most proposals violate the likelihood constraint or get rejected by the prior ratio.

So it is useful to do a small sensitivity check.

In [ ]:
def assess_proposal_scale(
    Y,
    Sigma,
    proposal_scales=(0.15, 0.3, 0.45, 0.8),
    n_rep=25,
    N_live=100,
    max_iter=500,
    mcmc_steps=80,
    seed=999
):
    true_logZ = true_log_evidence_general(Y, Sigma)
    rng = np.random.default_rng(seed)
    results = {}

    for scale in proposal_scales:
        estimates = []
        acceptances = []

        for _ in range(n_rep):
            run_seed = int(rng.integers(0, 10**9))
            out = nested_sampling_general_mcmc(
                Y,
                Sigma,
                N_live=N_live,
                max_iter=max_iter,
                seed=run_seed,
                mcmc_steps=mcmc_steps,
                proposal_scale=scale,
                return_path=False
            )
            estimates.append(out["logZ_hat"])
            acceptances.append(out["mean_mh_acceptance"])

        estimates = np.array(estimates)
        errors = estimates - true_logZ

        results[scale] = {
            "rmse": np.sqrt(np.mean(errors**2)),
            "mean_acceptance": np.mean(acceptances)
        }

    return results

In [ ]:
scale_results = assess_proposal_scale(
    Y,
    Sigma,
    proposal_scales=(0.15, 0.3, 0.45, 0.8),
    n_rep=25,
    N_live=100,
    max_iter=500,
    mcmc_steps=80,
    seed=999
)

for scale, res in scale_results.items():
    print(
        f"proposal_scale={scale:.2f} | "
        f"rmse={res['rmse']:.6f} | "
        f"mean_acceptance={res['mean_acceptance']:.3f}"
    )

In [ ]:
scales = sorted(scale_results.keys())
rmse_vals = [scale_results[s]["rmse"] for s in scales]
acc_vals = [scale_results[s]["mean_acceptance"] for s in scales]

plt.figure()
plt.plot(scales, rmse_vals, marker="o")
plt.xlabel("Proposal scale")
plt.ylabel("RMSE of log-evidence")
plt.title("Sensitivity to proposal scale")
plt.show()

plt.figure()
plt.plot(scales, acc_vals, marker="o")
plt.xlabel("Proposal scale")
plt.ylabel("Mean MH acceptance rate")
plt.title("Acceptance rate vs proposal scale")
plt.show()

## 13. Conclusion

The main message of this notebook is the following.

In the general case, we can no longer sample exactly from the constrained prior, but we can still implement nested sampling by replacing the exact draw with an MCMC draw targeting

$$
\pi(	\theta \mid L(	\theta) > \ell).
$$

The procedure is:

1. remove the worst live point,
2. define the likelihood threshold from that point,
3. start an MCMC chain from a surviving live point,
4. run a constrained Metropolis-Hastings kernel,
5. use the final state as the new live point.

This produces a practical **NS-MCMC algorithm**.

Compared with the special case, the main new difficulty is no longer the nested sampling formula itself, but rather the quality of the constrained MCMC exploration.  
That is why the two most important practical tuning parameters are:

- the number of MCMC steps per replacement,
- the proposal scale.